# Tes iMCLab: ON/OFF dan Kedap-Kedip via Python

**Pasangan sketch Arduino:** `kedap_kedip_Motor.ino` (folder *Pemrograman basic robotika dg Kit iMCLab*).

Sebelum menjalankan:
1. Upload `kedap_kedip_Motor.ino` ke ESP32 lewat kabel USB.
2. **Tutup Serial Monitor Arduino IDE** (satu port serial hanya bisa dipakai satu program).
3. Jalankan sel di bawah ini **dari atas ke bawah**, satu per satu (Shift+Enter).

Perintah yang dipahami sketch: `1` = ON, `0` = OFF, `b` = kedap-kedip.

In [1]:
# Sel 1 - Import & konfigurasi
# Kalau error "No module named serial": jalankan  !pip install pyserial
import serial
import serial.tools.list_ports
import time

BAUDRATE = 115200   # harus sama dengan Serial.begin() di ESP32
PORT = None         # None = cari otomatis. Atau isi manual, contoh: "COM6"

esp32 = None        # akan diisi di sel koneksi

In [2]:
# Sel 2 - Fungsi bantu (jalankan sekali)
ESP32_KEYWORDS = [
    "cp210", "cp2102", "cp210x", "ch340", "ch341", "ch34x",
    "ftdi", "ft232", "usb-serial", "usb serial", "usb uart",
    "uart bridge", "silicon labs", "espressif", "esp32",
]
ESP32_VID_PID = {
    (0x10C4, 0xEA60), (0x10C4, 0xEA70), (0x1A86, 0x7523),
    (0x1A86, 0x55D4), (0x0403, 0x6001), (0x0403, 0x6015),
    (0x303A, 0x1001),
}

def list_serial_ports():
    ports = serial.tools.list_ports.comports()
    if not ports:
        print("Tidak ada port serial yang terdeteksi.")
        return []
    print("Port serial yang tersedia:")
    for p in ports:
        print(f"  - {p.device} : {p.description}")
    return list(ports)

def deteksi_port_esp32():
    """Cari port ESP32 otomatis. Return nama port atau None."""
    kandidat = []
    for p in serial.tools.list_ports.comports():
        teks = f"{p.description} {p.manufacturer} {p.hwid}".lower()
        skor = 0
        if (p.vid, p.pid) in ESP32_VID_PID:
            skor += 100
        if any(kw in teks for kw in ESP32_KEYWORDS):
            skor += 50
        if "bluetooth" in teks:
            skor -= 200
        if skor > 0:
            kandidat.append((skor, p.device))
    if not kandidat:
        return None
    kandidat.sort(reverse=True)
    return kandidat[0][1]

def buka_koneksi(port, baudrate=BAUDRATE):
    """Buka port serial. Kalau sebelumnya masih terbuka, ditutup dulu."""
    global esp32
    if esp32 is not None and esp32.is_open:
        esp32.close()
    esp32 = serial.Serial(port, baudrate, timeout=0.2)
    time.sleep(2)                 # ESP32 reset otomatis saat port dibuka
    esp32.reset_input_buffer()    # buang pesan booting
    print(f"Terhubung ke ESP32 di {port} @ {baudrate} baud")

def baca_balasan(durasi=0.5):
    """Baca & tampilkan balasan ESP32 selama `durasi` detik."""
    akhir = time.time() + durasi
    while time.time() < akhir:
        baris = esp32.readline().decode("utf-8", errors="ignore").strip()
        if baris:
            print("ESP32:", baris)

def kirim(perintah, tunggu=0.5):
    """Kirim 1 karakter perintah ('1', '0', atau 'b') lalu tampilkan balasan."""
    esp32.write(perintah.encode("utf-8"))
    esp32.flush()
    baca_balasan(tunggu)

def tutup_koneksi():
    global esp32
    if esp32 is not None and esp32.is_open:
        kirim("0")        # pastikan motor mati
        esp32.close()
    print("Koneksi serial ditutup.")

## Langkah 1 - Cek port
Hasilnya (nama port, mis. `COM6`) bisa di-screenshot untuk laporan.

In [3]:
# Sel 3 - Tampilkan semua port dan cari port ESP32
list_serial_ports()
if PORT is None:
    PORT = deteksi_port_esp32()
print("\nPort ESP32 terpilih:", PORT)

Port serial yang tersedia:
  - COM6 : USB-Enhanced-SERIAL CH9102 (COM6)

Port ESP32 terpilih: COM6


## Langkah 2 - Buka koneksi
Kalau muncul `PermissionError` / *Access is denied*, tutup Serial Monitor Arduino IDE
(atau restart kernel: menu **Kernel -> Restart**), lalu jalankan lagi.

In [4]:
# Sel 4 - Buka koneksi ke ESP32
buka_koneksi(PORT)

Terhubung ke ESP32 di COM6 @ 115200 baud


## Langkah 3 - ON / OFF

In [5]:
# Sel 5 - Motor + LED ON 2 detik, lalu OFF
print(">> ON");  kirim("1")
time.sleep(2)
print(">> OFF"); kirim("0")

>> ON
ESP32: LED is on !!!
>> OFF
ESP32: LED is off !!!


## Langkah 4 - Kedap-kedip
Sketch akan kedap-kedip selama 5 detik (diatur `BLINK_DURATION_MS` di `.ino`), lalu mati sendiri.
Sel ini menunggu 7 detik supaya pesan awal dan akhir kedap-kedip sama-sama tampil.

In [6]:
# Sel 6 - Mulai kedap-kedip dan pantau pesan dari ESP32
print(">> Kedap-kedip")
kirim("b", tunggu=7)

>> Kedap-kedip
ESP32: Kedap-kedip dimulai selama 5.0 detik
ESP32: Kedap-kedip selesai. LED & motor OFF.


## Langkah 5 - Batalkan kedap-kedip di tengah jalan (opsional)
Jalankan sel ini, lalu **segera** jalankan sel berikutnya sebelum 5 detik habis.

In [ ]:
# Sel 7a - Mulai kedap-kedip tanpa menunggu
esp32.write(b"b")

In [ ]:
# Sel 7b - Hentikan sebelum waktunya
kirim("0")

## Langkah 6 - Tutup koneksi
Selalu jalankan ini di akhir supaya port bisa dipakai lagi.

In [ ]:
# Sel 8 - Tutup koneksi
tutup_koneksi()